# Logistic Regression Model
## AI3023 Machine Learning Workshop Course Project

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [ ]:
# 加载数据
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

y = train['Transported'].astype(int)
test_ids = test['PassengerId'].copy()
train_len = len(train)

all_data = pd.concat([train.drop('Transported', axis=1), test], axis=0).reset_index(drop=True)
print(f"数据加载完成: train={train_len}, test={len(test)}")

## 特征工程

In [ ]:
# 从PassengerId提取团队信息
all_data[['TeamId', 'Teamcode']] = all_data['PassengerId'].str.split('_', expand=True).astype(int)

# 从Cabin提取甲板、编号、侧边
all_data[['Deck', 'Num', 'Side']] = all_data['Cabin'].str.split('/', expand=True)

# 布尔值转换为0/1
all_data['CryoSleep'] = all_data['CryoSleep'].map({True: 1, False: 0})
all_data['VIP'] = all_data['VIP'].map({True: 1, False: 0})

# 团队大小
all_data['GroupSize'] = all_data['TeamId'].map(all_data['TeamId'].value_counts())

# 是否独行
all_data['Name'] = all_data['Name'].fillna('Unknown')
all_data['Lastname'] = all_data['Name'].str.split().str[-1]
all_data['FamilySize'] = all_data['Lastname'].map(all_data['Lastname'].value_counts())
all_data['Is_Solo'] = ((all_data['GroupSize'] == 1) & (all_data['FamilySize'] == 1)).astype(int)

## 缺失值处理

In [ ]:
# CryoSleep和VIP的缺失值用0填充
all_data['CryoSleep'] = all_data['CryoSleep'].fillna(0)
all_data['VIP'] = all_data['VIP'].fillna(0)

# HomePlanet用众数填充
all_data['HomePlanet'] = all_data['HomePlanet'].fillna(all_data['HomePlanet'].mode()[0])

# Destination用众数填充
all_data['Destination'] = all_data['Destination'].fillna(all_data['Destination'].mode()[0])

# Age用中位数填充
all_data['Age'] = all_data['Age'].fillna(all_data['Age'].median())

# 消费列用0填充
col_spend = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in col_spend:
    all_data[col] = all_data[col].fillna(0)

# 添加总消费特征
all_data['Total_Spend'] = all_data[col_spend].sum(axis=1)

# Deck和Side用众数填充
all_data['Deck'] = all_data['Deck'].fillna(all_data['Deck'].mode()[0])
all_data['Side'] = all_data['Side'].fillna(all_data['Side'].mode()[0])

# Num转换为数字，缺失值用中位数填充
all_data['Num'] = pd.to_numeric(all_data['Num'], errors='coerce')
all_data['Num'] = all_data['Num'].fillna(all_data['Num'].median())

## 编码

In [ ]:
# 对分类变量进行独热编码
cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
all_data = pd.get_dummies(all_data, columns=cat_cols)

# 删除不需要的列
drop_cols = ['PassengerId', 'Cabin', 'Name', 'Lastname', 'TeamId', 'Teamcode']
all_data = all_data.drop(columns=drop_cols)

print(f"特征数量: {all_data.shape[1]}")
print(f"是否有缺失值: {all_data.isnull().sum().sum()}")

## 准备训练集和测试集

In [ ]:
# 划分训练集和测试集
X = all_data.iloc[:train_len]
X_test = all_data.iloc[train_len:]

print(f"训练集形状: {X.shape}")
print(f"测试集形状: {X_test.shape}")

## 超参数调优

In [ ]:
# 手动尝试不同的C值（正则化强度）
best_score = 0
best_C = 1

for C in [0.1, 1, 10]:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    mean_score = scores.mean()
    print(f"C={C}: 准确率={mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_C = C

print(f"\n最佳C值: {best_C}, 最佳准确率: {best_score:.4f}")

## 训练模型并预测

In [ ]:
# 使用最佳参数训练模型
final_model = LogisticRegression(C=best_C, max_iter=1000, random_state=42)
final_model.fit(X, y)

# 预测
test_preds = final_model.predict(X_test)

# 保存结果
output = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': test_preds.astype(bool)
})
output.to_csv('submission_lr.csv', index=False)

print(f"完成！结果保存为: submission_lr.csv")
print(f"预测分布:")
print(output['Transported'].value_counts())